In [1]:
! gdown 1kEelcRchc3iQMVS9ALW3ETIMJtD0K5bp

Downloading...
From: https://drive.google.com/uc?id=1kEelcRchc3iQMVS9ALW3ETIMJtD0K5bp
To: /kaggle/working/cultural_dataset_combined_104.json
100%|█████████████████████████████████████████| 158k/158k [00:00<00:00, 128MB/s]


In [2]:
%%capture
! pip install transformers accelerate bitsandbytes

In [3]:
import os
import json
import math
import argparse
from typing import List, Dict, Any

import torch
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from huggingface_hub import login
import shlex


In [4]:
system_prompt = """You are a medical expert assistant.
You are given multiple-choice medical questions.
Your task is to analyze the question and the provided answer options, apply correct medical reasoning, and choose the single best answer.

OUTPUT CONSTRAINTS (STRICT):
- The explanation must be exactly 2–3 sentences.
- Do NOT use bullet points, numbering, or headings.
- Do NOT include disclaimers, preambles, or meta-comments.
- Do NOT include extra whitespace or blank lines.
- The response must follow the exact format described below.

PENALTIES FOR VIOLATION:
- Any deviation from the required format, sentence count, or final answer line will be considered a critical error.
- Any text appearing after the final answer line will invalidate the response.
- Failure to end with the exact phrase will result in a zero score.

FINAL FORMAT (MANDATORY):
- After the explanation, output exactly one final line:
"The answer is X"
where X is one of (A, B, C, D, or E)."""

user_prompt = """Question:
{question}

Options:
{options}

Provide a medical explanation in exactly 2–3 sentences.
Strictly follow all formatting rules.
Then end the response with exactly this line and nothing else:
"The answer is X"
where X is the correct option letter."""


def build_messages(question_text: str, options: str) -> List[Dict[str, str]]:

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": user_prompt.format(
                question=question_text,
                options=options
            ),
        },
    ]
    return messages


def load_model_and_tokenizer(model_name: str, use_4bit: bool = True):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # Many causal LMs need a pad token for batched generation.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        
    tokenizer.padding_side = "left"
        
    if use_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(
                torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
                else torch.float16
            ),
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        dtype = (
            torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
            else torch.float16 if torch.cuda.is_available()
            else torch.float32
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )

    model.eval()
    return model, tokenizer


@torch.inference_mode()
def batched_generate(
    model,
    tokenizer,
    messages_batch: List[List[Dict[str, str]]],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> List[str]:
    """
    Batched chat generation using the tokenizer chat template.
    """
    # Convert chat messages to text prompts using the model's chat template.
    prompts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in messages_batch
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    # Move batch tensors to the model device.
    # For quantized models with device_map="auto", input tensors should go to the
    # first device used by the model.
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    do_sample = temperature > 0.0

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Remove the prompt tokens so only newly generated text remains.
    generated_only = outputs[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

    return [x.strip() for x in decoded]


def main(args):
    if args.hf_token:
        login(token=args.hf_token)

    with open(args.input_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    model, tokenizer = load_model_and_tokenizer(
        model_name=args.model_name,
        use_4bit=not args.disable_4bit,
    )

    if args.culture_mode:
        all_messages = [
            build_messages(
                d["w culture question"],
                d["options"]
            )
            for d in dataset
        ]
    else:
        all_messages = [
            build_messages(
                d["w/o culture question"],
                d["options"]
            )
            for d in dataset
        ]

    all_answers = []

    for start in tqdm(range(0, len(all_messages), args.batch_size), desc="Generating"):
        batch_messages = all_messages[start:start + args.batch_size]

        batch_outputs = batched_generate(
            model=model,
            tokenizer=tokenizer,
            messages_batch=batch_messages,
            max_new_tokens=args.max_new_tokens,
            temperature=args.temperature,
            top_p=args.top_p,
        )
        all_answers.extend(batch_outputs)

        # Optional: free a bit of cached memory between batches on notebook GPUs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    with open(args.output_path, "w", encoding="utf-8") as f:
        json.dump(all_answers, f, ensure_ascii=False, indent=2)

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Run batched quantized inference on cultural MCQA data.")
    parser.add_argument(
        "--input_path",
        type=str,
        default="/kaggle/working/cultural_dataset_combined_104.json",
        help="Path to input JSON file.",
    )
    parser.add_argument(
        "--output_path",
        type=str,
        default="/kaggle/working/culture_mcqs_answers.json",
        help="Path to output JSON file.",
    )
    parser.add_argument(
        "--model_name",
        type=str,
        default="google/medgemma-4b-it",
        help="HF model name.",
    )
    parser.add_argument(
        "--hf_token",
        type=str,
        default="",
        help="HF token. Prefer passing via env var or notebook login, not hard-coding.",
    )
    parser.add_argument(
        "--culture_mode",
        action='store_true',
        help='Optional role, e.g. "Japan" or "Arabic".',
    )
    parser.add_argument(
        "--max_new_tokens",
        type=int,
        default=4096,
        help="Maximum new tokens to generate.",
    )
    parser.add_argument(
        "--batch_size",
        type=int,
        default=4,
        help="Batch size for generation. Try 2/4/8 depending on GPU memory.",
    )
    parser.add_argument(
        "--temperature",
        type=float,
        default=0.0,
        help="Set >0 for sampling. 0.0 = greedy decoding.",
    )
    parser.add_argument(
        "--top_p",
        type=float,
        default=1.0,
        help="Top-p for sampling.",
    )
    parser.add_argument(
        "--disable_4bit",
        action="store_true",
        help="Disable 4-bit quantization.",
    )

    # arg_string = "--model_name meta-llama/Llama-3.1-8B-Instruct \
    # --culture_mode \
    # --output_path Llama-3.1-8B-Instruct_yes_results.jsonl"
    
    # args = parser.parse_args(shlex.split(arg_string))
    # main(args)

    arg_string = "--model_name meta-llama/Llama-3.1-8B-Instruct \
    --output_path Llama-3.1-8B-Instruct_no_results.jsonl"
    
    args = parser.parse_args(shlex.split(arg_string))
    main(args)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Generating:   0%|          | 0/26 [00:00<?, ?it/s]